# Artificial and Computational Intelligence Assignment I

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


 ## Group No 227
### **Group Members:**
|Sl No.|BITS ID| Name|
|:---:|:--:|------:|
|1|2025aa05224|SRIVYSHNAV K S|
|2|2025aa05044|SUVRADIP PAUL|
|3|2025aa05081|THAMIZHCHELVAN G|
|4|2025aa05303|UTKARSH SRIVASTAVA|
|5|2025aa05078|UTTAM KUMAR|

### Problem Statement

After a major earthquake, an autonomous rescue agent is deployed into an 8x8 grid environment to locate and rescue three trapped survivors.
The robot must be programmed using a Greedy Best-First Search (GBFS) algorithm to dynamically navigate from its drop-off point (Start Node)
to the closest survivor (Goal Node), while strictly avoiding collapsed structures and minimizing its exposure to toxic gas zones

### 1. PEAS for the Problem
#### PEAS Description :
* **Performance (P) :** 1.) Rescure all survivors, 2.) Minimize total path cost, 3.) Minimize traversal through given toxic zones, 4.) Minimize time in action
* **Environment (E) :** 1.) 8*8 Grid, 2.) Single Agent, 3.) Discrete in nature, 4.) Fully observable for survivors
* **Actuators (A) :** 1.) Orthogonally movement, 2.) Safe Passage markers, 3.) Agent's Hand
* **Sensor (S) :** 1.) Scanners, 2.) Identify cell state

In [17]:
# Importing required python libs.
import heapq
import time
import os

### GBFS Algorithm implementation
#### Quick Ref Link for GBFS :- https://www.youtube.com/watch?v=GVvN0ikNekw

In [18]:
##This function (gbfs_search_algo) implements the Greedy Best-First Search (GBFS) algorithm to find a path from a start node to a goal node
##using only on heuristic function f(n) = h(n).

def gbfs_search_algo(agent_ra, start_node, goal_pos, h_func):
        pq = []
        counter = 0

        # Passed heuristic function has been used to calculate nodes and added into heap queue for traveling and tracking purpose
        h_start = h_func(start_node[0], start_node[1], goal_pos[0], goal_pos[1])
        heapq.heappush(pq, (h_start, counter, start_node, [start_node]))

        explored = set()
        toxic_crossed = 0

        while pq:
            frontier_nodes = [(h, node) for h, _, node, _ in pq]
            agent_ra.log(f"Frontier nodes tracking under GBFS_algo search: {frontier_nodes}")

            #Poping up for nodes which as been visted or explored and added to explored set.
            f, _, current, path = heapq.heappop(pq)
            if current in explored:
                continue
            explored.add(current)
            agent_ra.log(f"Explored nodes tracking under GBFS_algo search: {current}")

            # Validating Goal box has been reached or not; if reached then stop and return value with toxic zones accumulated values.
            if current == goal_pos:
                # Below condition validating and calculating toxic zones values.
                for r, c in path:
                    if agent_ra.grid[r][c] == 'T': toxic_crossed += 1
                return path, len(explored), toxic_crossed

            # if Goal has been reached then visit neighbors nodes to get the target goal.
            for nxt in agent_ra.get_neighbors(current[0], current[1]):
                if nxt not in explored:
                    counter += 1
                    #As per algo calculate the heuristic value for next nodes lookup and pushing into heap queue.
                    h_val = h_func(nxt[0], nxt[1], goal_pos[0], goal_pos[1])
                    agent_ra.log(f"Calculated heuristic for {nxt}: {h_val} from GBFS_algo search")
                    heapq.heappush(pq, (h_val, counter, nxt, path + [nxt]))
        # Negative boundary condition; if path not found then.
        return [], len(explored), 0 # Path not found


### A* Algorithm implementation
#### Quick Ref Link for A* Algo :- https://www.youtube.com/watch?v=Eiu1Cb-veCc

In [19]:
##This function (astar_search_algo) implements the A* algorithm (best in BFS search category) to find a path from a start node to a goal node
##using a heuristic function and actual cost applied to reach the nodes f(n) = g(n) + h(n).

def astar_search_algo(agent_ra, start_node, goal_pos, h_func):
        """Executes A* Search to find a specific goal (f = g + h)."""
        pq = []
        counter = 0

        # Passed heuristic function has been used to calculate nodes and added into heap queue for traveling and tracking purpose
        h_start = h_func(start_node[0], start_node[1], goal_pos[0], goal_pos[1])
        heapq.heappush(pq, (h_start, counter, start_node, [start_node], 0))

        explored = set()
        toxic_crossed = 0

        while pq:
            # Deliverable (d): Frontier Tracking
            frontier_nodes = [(f, node) for f, _, node, _, _ in pq]
            agent_ra.log(f"Frontier nodes tracking under A*_algo search: {frontier_nodes}")

            f_val, _, current, path, g = heapq.heappop(pq)

            if current in explored:
                continue

            explored.add(current)
            agent_ra.log(f"Explored nodes tracking under A*_algo search: {current}")

            # if Goal has been reached then visit neighbors nodes to get the target goal.
            if current == goal_pos:
                for r, c in path:
                    if agent_ra.grid[r][c] == 'T': toxic_crossed += 1
                return path, len(explored), toxic_crossed

            for nxt in agent_ra.get_neighbors(current[0], current[1]):
                if nxt not in explored:
                    counter += 1

                    #As per algo calculate the heuristic value for next nodes lookup and pushing into heap queue.
                    h_val = h_func(nxt[0], nxt[1], goal_pos[0], goal_pos[1])
                    agent_ra.log(f"Calculated heuristic for {nxt}: {h_val} from A*_algo search")
                    g_new = g + 1
                    f_new = g_new + h_val
                    heapq.heappush(pq, (f_new, counter, nxt, path + [nxt], g_new))

        return [], len(explored), 0


In [20]:

class RescueAgent:
    def __init__(self, env_filename, out_filename):
        self.grid = []
        self.start = None
        self.survivors = {}
        self.env_filename = env_filename
        self.out_filename = out_filename
        self.load_environment()

    def log(self, message=""):
        """Prints to console AND writes to the output file."""
        print(message)
        with open(self.out_filename, 'a') as f:
            f.write(str(message) + '\n')

    def load_environment(self):
        """Loads the environment grid and parameters from the external file."""
        if not os.path.exists(self.env_filename):
            self.log(f"CRITICAL ERROR: '{self.env_filename}' not found!")
            return

        # Dictionary to store the variables extracted from the text file
        env_vars = {}

        try:
            with open(self.env_filename, 'r') as file:
                # Read the entire file and execute it to extract the variables into env_vars
                exec(file.read(), {}, env_vars)

            # 1. Load the Grid
            self.grid = env_vars.get('RAW_GRID', [])

            # 2. Load the Start Node
            self.start = env_vars.get('START_NODE', None)

            # 3. Load the Survivors
            # The input provides a list of tuples: [(4, 7), (6, 4), (7, 5)]
            # We must map them into our dictionary format: {'1': (4, 7), '2': (6, 4), ...}
            survivors_list = env_vars.get('SURVIVORS', [])
            self.survivors = {}
            for idx, pos in enumerate(survivors_list):
                self.survivors[str(idx + 1)] = pos

        except Exception as e:
            self.log(f"ERROR parsing the input file: {e}")

    def print_grid(self):
        """Visually represents the current state of the grid."""
        for row in self.grid:
            self.log(" ".join(row))
        self.log("")

    # ---------------------------------------------------------
    # HEURISTICS & MOVEMENT LOGIC
    # ---------------------------------------------------------
    def get_neighbors(self, r, c):
        """Returns valid strictly ORTHOGONAL neighbors."""
        neighbors = []
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            # Ensure within bounds and not blocked (#)
            if 0 <= nr < len(self.grid) and 0 <= nc < len(self.grid) and self.grid[nr][nc] != '#':
                neighbors.append((nr, nc))
        return neighbors

    def heuristic_1(self, r, c, goal_r, goal_c):
        """H1: Pure Manhattan Distance"""
        return abs(r - goal_r) + abs(c - goal_c)

    def heuristic_2(self, r, c, goal_r, goal_c):
        """H2: Risk-Aware Heuristic -> h1(n) + alpha penalty if node is Toxic"""
        h1 = self.heuristic_1(r, c, goal_r, goal_c)
        alpha = 2
        # Apply penalty of alpha (2) if traversing through a Toxic zone
        penalty = alpha if self.grid[r][c] == 'T' else 0
        return h1 + penalty

    def select_next_target(self, current_pos, remaining_survivors, h_func):
        """Finds the closest survivor based on the selected heuristic."""
        best_goal, best_h = None, float('inf')
        for g_id, g_pos in remaining_survivors.items():
            # FIX: Properly extract the row and column for both tuples
            h_val = h_func(current_pos[0], current_pos[1], g_pos[0], g_pos[1])
            if h_val < best_h:
                best_h = h_val
                best_goal = g_id
        return best_goal, remaining_survivors[best_goal]

    # ---------------------------------------------------------
    # MISSION EXECUTION LOOP
    # ---------------------------------------------------------
    def run_mission(self, algo='GBFS', h_type=1):
        curr = self.start
        if not curr:
            self.log("ERROR: Start position 'S' not found.")
            return

        rem_survivors = dict(self.survivors)
        h_func = self.heuristic_1 if h_type == 1 else self.heuristic_2

        total_explored = 0
        total_toxic = 0
        rescue_seq = []

        self.log(f"\n==================================================")
        self.log(f"RUNNING MISSION: {algo} with Heuristic {h_type}")
        self.log(f"==================================================")

        start_time = time.time()

        while rem_survivors:
            # Deliverable (g): Select Next Node
            target_id, target_pos = self.select_next_target(curr, rem_survivors, h_func)
            self.log(f"\n[Targeting Survivor {target_id} at {target_pos}]")
            self.log(f"g. Selected Next Node: Survivor {target_id}")

            if algo == 'GBFS':
                path, explored_count, toxic = gbfs_search_algo(self, curr, target_pos, h_func)
            else:
                path, explored_count, toxic = astar_search_algo(self, curr, target_pos, h_func)

            if not path:
                self.log(f"Survivor {target_id} is unreachable! (Trap/Dead-End)")
                del rem_survivors[target_id]
                continue

            total_explored += explored_count
            total_toxic += toxic

            # Deliverable (h, i, k): Print Stats
            self.log(f"\nFinal Path: {path}")
            self.log(f"\nToxic zones crossed: {toxic}")
            self.log(f"\nTries (Nodes Explored): {explored_count}")

            rescue_seq.append(target_id)
            curr = path[-1]

            # Deliverable (j & 9b): Mark node as safe passage & Update Grid
            self.grid[curr[0]][curr[1]] = '.'
            del rem_survivors[target_id]
            self.log(f"\nUpdated Grid after rescuing Survivor {target_id}:")
            self.print_grid()

        exec_time = (time.time() - start_time) * 1000

        # Deliverable (4 & 10): Complexity Analysis Output
        self.log(f"*"*50)
        self.log(f"--- MISSION SUMMARY ({algo} - H{h_type}) ---")
        self.log(f"*"*50)
        self.log(f"\nRescue Sequence: {rescue_seq}")
        self.log(f"\nTotal Nodes Explored: {total_explored}")
        self.log(f"\nTotal Toxic Zones Crossed: {total_toxic}")
        self.log(f"\nExecution Time: {exec_time:.2f} ms\n")


if __name__ == "__main__":
    env_file = "inputPS03.txt"
    out_file = "outputPS03.txt"

    # 1. Clear/Initialize the output file at the start of execution
    with open(out_file, 'w') as f:
        f.write("=== Rescue Robot Assignment Execution Log ===\n\n")

    # 2. Safety Check: Verify input file exists
    if not os.path.exists(env_file):
        print(f"ERROR: '{env_file}' not found. Please create it in the same folder.")
    else:
        # Initial Grid Printing
        agent_init = RescueAgent(env_file, out_file)
        agent_init.log("c. Initial Grid (Loaded from File):")
        agent_init.print_grid()

        # Run GBFS with Heuristic 1
        agent_h1 = RescueAgent(env_file, out_file)
        agent_h1.run_mission(algo='GBFS', h_type=1)

        # Run GBFS with Heuristic 2
        agent_h2 = RescueAgent(env_file, out_file)
        agent_h2.run_mission(algo='GBFS', h_type=2)

        # Compare with A* using Heuristic 1
        agent_astar = RescueAgent(env_file, out_file)
        agent_astar.run_mission(algo='A*', h_type=1)

        print(f"\nExecution finished successfully. Check {out_file} for the output log.")


c. Initial Grid (Loaded from File):
S . . # . T . .
# . T # . . . .
. . . . . T . #
. . # # . . . .
T . . . . # # P
. . . . T . . .
. . # . . P # T
. . . . T . P .


RUNNING MISSION: GBFS with Heuristic 1

[Targeting Survivor 2 at (6, 4)]
g. Selected Next Node: Survivor 2
Frontier nodes tracking under GBFS_algo search: [(10, (0, 0))]
Explored nodes tracking under GBFS_algo search: (0, 0)
Calculated heuristic for (0, 1): 9 from GBFS_algo search
Frontier nodes tracking under GBFS_algo search: [(9, (0, 1))]
Explored nodes tracking under GBFS_algo search: (0, 1)
Calculated heuristic for (1, 1): 8 from GBFS_algo search
Calculated heuristic for (0, 2): 8 from GBFS_algo search
Frontier nodes tracking under GBFS_algo search: [(8, (1, 1)), (8, (0, 2))]
Explored nodes tracking under GBFS_algo search: (1, 1)
Calculated heuristic for (2, 1): 7 from GBFS_algo search
Calculated heuristic for (1, 2): 7 from GBFS_algo search
Frontier nodes tracking under GBFS_algo search: [(7, (2, 1)), (8, (0, 2)), (7

In [21]:
import os
print(os.getcwd())

/content
